> Resetto le variabili.

In [ ]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])? N
Nothing done.


> Importo moduli.



In [3]:
import os
import glob
import nibabel
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.decomposition import PCA
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


# PRE-PROCESSING

> Variabili utili.

In [57]:
image_size = np.array([61,73,61])

> Creo matrice dei pazienti.

In [58]:
os.chdir('/content/drive/My Drive/Brain - Tatiana/fALFF/4') 
images_pazienti = glob.glob('*.nii', recursive=True)
num_pazienti = len(images_pazienti) 

X_pazienti = np.zeros((num_pazienti,np.product(image_size)))
t = 0
for fMRI in images_pazienti:
  file_nii = nibabel.load(fMRI)
  img = np.array(file_nii.dataobj)
  X_pazienti[t,:] = np.reshape(img,(1,np.product(image_size)))
  t = t + 1

y_pazienti = np.ones((num_pazienti,1))

In [59]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


> Creao matrice dei controlli.

In [60]:
os.chdir('/content/drive/My Drive/Brain - Tatiana/fALFF/5') 
images_controlli = glob.glob('*.nii', recursive=True) 
num_controlli = len(images_controlli)

X_controlli = np.zeros((num_controlli,np.product(image_size)))
t = 0
for fMRI in images_controlli:
  file_nii = nibabel.load(fMRI)
  img = np.array(file_nii.dataobj)
  X_controlli[t,:] = np.reshape(img,(1,np.product(image_size)))
  t = t + 1

y_controlli = np.zeros((num_controlli,1))

> Unisco le matrici.

In [113]:
X = np.concatenate((X_pazienti,X_controlli))
y = np.concatenate((y_pazienti,y_controlli))
y_n = 1-y

Elimino dalla matrice X le colonne con somma 0 (i.e. le colonne corrispondenti a voxel neri in tutti i soggetti)

In [114]:

initial_num_cols = X.shape[1]

mask = (X == 0).all(0)
column_indices = np.where(mask)[0]
X = X[:,~mask]

final_num_cols = X.shape[1]

print(str(initial_num_cols - final_num_cols) + ' columns were dropped from the dataset')


200802 columns were dropped from the dataset


# PCA

In [116]:
scaler1 = StandardScaler()
scaler1.fit(X)
X_scaled = scaler1.transform(X)

pca = PCA(n_components=25)
pca.fit(X_scaled)
X_pca = pca.transform(X_scaled)
#X_pca = X_scaled

In [117]:
X.shape

(34, 70831)

# Parameter estimation

In [118]:
# Set the parameters by cross-validation
#tuned_parameters = [{'kernel': ['linear','sigmoid'], 'gamma': [1e-2,1e-3,1e-4],
#                     'C': [.1,1, 10, 100, 1000]}]
#tuned_parameters = [{'kernel': ['linear'], 
#                     'C': [.1, 1, 10, 100, 1000, 5000,10000]}]
tuned_parameters = [{'kernel': ['rbf','linear','sigmoid','poly'], 'gamma': [1e-3, 1e-4],
                     'C': [1, 10, 100, 1000]}]

clf = GridSearchCV(svm.SVC(), tuned_parameters, scoring='accuracy', cv=LeaveOneOut())
clf.fit(X_pca, y.ravel())

print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
print("Grid scores on development set:")
print()
means = clf.cv_results_['mean_test_score']
stds = clf.cv_results_['std_test_score']
for mean, std, params in zip(means, stds, clf.cv_results_['params']):
        print("%0.3f (+/-%0.03f) for %r"
              % (mean, std * 2, params))

Best parameters set found on development set:

{'C': 1000, 'gamma': 0.001, 'kernel': 'sigmoid'}

Grid scores on development set:

0.000 (+/-0.000) for {'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}
0.353 (+/-0.956) for {'C': 1, 'gamma': 0.001, 'kernel': 'linear'}
0.235 (+/-0.848) for {'C': 1, 'gamma': 0.001, 'kernel': 'sigmoid'}
0.441 (+/-0.993) for {'C': 1, 'gamma': 0.001, 'kernel': 'poly'}
0.059 (+/-0.471) for {'C': 1, 'gamma': 0.0001, 'kernel': 'rbf'}
0.353 (+/-0.956) for {'C': 1, 'gamma': 0.0001, 'kernel': 'linear'}
0.441 (+/-0.993) for {'C': 1, 'gamma': 0.0001, 'kernel': 'sigmoid'}
0.441 (+/-0.993) for {'C': 1, 'gamma': 0.0001, 'kernel': 'poly'}
0.000 (+/-0.000) for {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
0.353 (+/-0.956) for {'C': 10, 'gamma': 0.001, 'kernel': 'linear'}
0.265 (+/-0.882) for {'C': 10, 'gamma': 0.001, 'kernel': 'sigmoid'}
0.441 (+/-0.993) for {'C': 10, 'gamma': 0.001, 'kernel': 'poly'}
0.088 (+/-0.567) for {'C': 10, 'gamma': 0.0001, 'kernel': 'rbf'}
0.353 (+/-0.956) 

# Leave-one out

In [119]:
# evaluate model
best_clf = clf.best_estimator_
y_pred=best_clf.predict(X_pca)
scores = cross_val_score(best_clf, X_pca, y.ravel(), scoring='accuracy', cv=LeaveOneOut(), n_jobs=-1)
# report performance
print(scores)
print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

[1. 0. 1. 1. 1. 0. 1. 1. 0. 1. 0. 0. 1. 0. 1. 1. 0. 0. 1. 1. 1. 1. 0. 1.
 0. 0. 0. 1. 0. 0. 1. 0. 1. 1.]
Accuracy: 0.559 (0.497)


In [120]:
print(scores)
print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

[1. 0. 1. 1. 1. 0. 1. 1. 0. 1. 0. 0. 1. 0. 1. 1. 0. 0. 1. 1. 1. 1. 0. 1.
 0. 0. 0. 1. 0. 0. 1. 0. 1. 1.]
Accuracy: 0.559 (0.497)
